# Road Damage Detection — Pothole / Crack / Manhole
YOLOv8-based detector with a severity/damage-scoring layer on top.

**What changed vs. your previous notebook (and why it wasn't predicting anything):**
1. The inference cell loaded `models/road_damage_yolov8.pt` — a path that never existed. It never loaded your trained weights, so predictions were meaningless. Fixed to load `best.pt` from the actual training run.
2. The `analyze_road_damage_photo` function was cut off mid-way (unclosed dict, no return). It couldn't have run at all. Completed and cleaned up.
3. Switched `yolov8n` (nano) → `yolov8s` (small). With ~2,000 images and 3 visually-overlapping classes, nano tends to underfit — `s` roughly doubles capacity for a modest speed cost, still fast enough for a FastAPI backend.
4. Train with `rect=True` — your images are 640×360 (16:9), and square-resizing them for training distorts thin/elongated cracks. Rectangular training keeps native aspect ratio.
5. Proper stratified-ish split with a fixed seed + a held-out **test** set (train/val/test), so your val metrics aren't the only thing you ever check.
6. Class-aware confidence thresholds kept (manhole is rare and safety-critical → lower threshold), but now actually wired to the right model.

**On "hole size":** with a single 2D photo and no camera calibration (known focal length + camera height, or a reference object of known size in frame), you cannot recover true physical size (cm) — only *relative* size (% of the image the damage covers, and pixel width/height). This notebook reports relative size honestly. If you need real-world cm, see the note at the bottom — it needs one extra piece of info you don't currently collect (mounting height or a reference marker).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

In [ ]:
import os, random, shutil, json
from collections import defaultdict

DATA_ROOT = "/content/drive/MyDrive/CivicPulse/Road_Damage/data"
IMAGES_DIR = f"{DATA_ROOT}/images"
LABELS_DIR = f"{DATA_ROOT}/labels-YOLO"

CLASS_NAMES = {0: 'pothole', 1: 'crack', 2: 'manhole'}
OUT = '/content/road_damage_split'

## 1. Split the data (stratified by dominant class, train/val/test)
Using a fixed seed so the split is reproducible run to run.

In [ ]:
random.seed(42)

all_images = sorted(f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png')))

def dominant_class(img_file):
    """Class that appears most often in this image's label file — used to stratify the split
    so train/val/test each get a fair share of the rare manhole class."""
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = f'{LABELS_DIR}/{label_file}'
    if not os.path.exists(label_path):
        return -1
    counts = defaultdict(int)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return max(counts, key=counts.get) if counts else -1

buckets = defaultdict(list)
for img in all_images:
    buckets[dominant_class(img)].append(img)

train_files, val_files, test_files = [], [], []
for cls, files in buckets.items():
    random.shuffle(files)
    n = len(files)
    n_train = int(n * 0.75)
    n_val = int(n * 0.15)
    train_files += files[:n_train]
    val_files += files[n_train:n_train + n_val]
    test_files += files[n_train + n_val:]

random.shuffle(train_files); random.shuffle(val_files); random.shuffle(test_files)

if os.path.exists(OUT):
    shutil.rmtree(OUT)

for split, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)
    for f in files:
        shutil.copy(f'{IMAGES_DIR}/{f}', f'{OUT}/images/{split}/{f}')
        label_file = os.path.splitext(f)[0] + '.txt'
        label_path = f'{LABELS_DIR}/{label_file}'
        if os.path.exists(label_path):
            shutil.copy(label_path, f'{OUT}/labels/{split}/{label_file}')

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

## 2. Sanity checks
Corrupt images and out-of-range coordinates would silently wreck training.

In [ ]:
from PIL import Image

corrupt = []
for split in ['train', 'val', 'test']:
    for fname in os.listdir(f'{OUT}/images/{split}'):
        try:
            img = Image.open(f'{OUT}/images/{split}/{fname}')
            img.verify()
        except Exception:
            corrupt.append((split, fname))
print(f"Corrupt images found: {len(corrupt)}")
if corrupt:
    print(corrupt[:10])

invalid_labels = []
for split in ['train', 'val', 'test']:
    for fname in os.listdir(f'{OUT}/labels/{split}'):
        with open(f'{OUT}/labels/{split}/{fname}') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                vals = list(map(float, parts[1:5]))
                if any(v < 0 or v > 1 for v in vals):
                    invalid_labels.append((split, fname))
print(f"Label files with out-of-range coordinates: {len(set(invalid_labels))}")

## 3. Quick class balance check per split
Make sure manhole (the rare class) actually landed in all three splits.

In [ ]:
for split in ['train', 'val', 'test']:
    counts = defaultdict(int)
    for fname in os.listdir(f'{OUT}/labels/{split}'):
        with open(f'{OUT}/labels/{split}/{fname}') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[CLASS_NAMES[int(parts[0])]] += 1
    print(split, dict(counts))

## 4. data.yaml

In [ ]:
yaml_content = f"""
path: {OUT}
train: images/train
val: images/val
test: images/test
names:
  0: pothole
  1: crack
  2: manhole
"""

with open(f'{OUT}/data.yaml', 'w') as f:
    f.write(yaml_content)

print(open(f'{OUT}/data.yaml').read())

## 5. Train

Notes on the choices below:
- `yolov8s.pt` instead of `yolov8n.pt` — more capacity, still trains in well under an hour on a Colab T4 for a dataset this size.
- `rect=True` — trains on the native 16:9 aspect ratio instead of squashing to square, which helps thin/elongated cracks.
- `epochs=150` with `patience=20` — nano runs often stop too early relative to what a larger backbone needs; early stopping will cut this short once val mAP plateaus, so you're not wasting time either way.
- `degrees`/`shear`/`perspective` augmentation kept modest — road photos are already taken from a fairly consistent angle, so heavy geometric augmentation mostly adds noise here rather than useful variety.


In [ ]:
model = YOLO('yolov8s.pt')

results = model.train(
    data=f'{OUT}/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    rect=True,
    patience=20,
    optimizer='auto',
    lr0=0.01,
    degrees=5,
    shear=2,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.8,
    project='/content/drive/MyDrive/road_damage/runs',
    name='road_damage_v2'
)

RUN_DIR = results.save_dir
BEST_WEIGHTS = f'{RUN_DIR}/weights/best.pt'
print("Best weights saved at:", BEST_WEIGHTS)

## 6. Validate on the held-out val split

In [ ]:
val_model = YOLO(BEST_WEIGHTS)
metrics = val_model.val(data=f'{OUT}/data.yaml', split='val')

print("Overall mAP50-95:", metrics.box.map)
print("Overall mAP50:", metrics.box.map50)

class_order = ['pothole', 'crack', 'manhole']
for i, name in enumerate(class_order):
    print(f"{name}: mAP50 = {metrics.box.maps[i]:.3f}")

In [ ]:
from IPython.display import Image as IPImage, display

print("Training/validation loss + mAP curves:")
display(IPImage(f'{RUN_DIR}/results.png'))

print("Confusion matrix — check manhole's row/column specifically:")
display(IPImage(f'{RUN_DIR}/confusion_matrix.png'))

print("Precision-Recall curve per class:")
display(IPImage(f'{RUN_DIR}/PR_curve.png'))

print("Sample validation batch with predicted boxes overlaid:")
display(IPImage(f'{RUN_DIR}/val_batch0_pred.jpg'))

## 7. Final check on the untouched test split
This is the number that tells you how the model will actually behave on new uploads.

In [ ]:
test_metrics = val_model.val(data=f'{OUT}/data.yaml', split='test')
print("Test mAP50:", test_metrics.box.map50)
print("Test mAP50-95:", test_metrics.box.map)
for i, name in enumerate(class_order):
    print(f"{name}: mAP50 = {test_metrics.box.maps[i]:.3f}")

## 8. Export for your FastAPI backend

In [ ]:
export_model = YOLO(BEST_WEIGHTS)
onnx_path = export_model.export(format='onnx')
print("ONNX model at:", onnx_path)

# Also copy the raw .pt to a stable, predictable path so your backend code
# doesn't need to know the training run's folder name.
STABLE_WEIGHTS_PATH = f'{DATA_ROOT}/../models/road_damage_yolov8s.pt'
os.makedirs(os.path.dirname(STABLE_WEIGHTS_PATH), exist_ok=True)
shutil.copy(BEST_WEIGHTS, STABLE_WEIGHTS_PATH)
print("Stable copy at:", STABLE_WEIGHTS_PATH)

## 9. Inference + damage scoring

Fixed version of your `analyze_road_damage_photo` — it now:
- loads the model you actually trained (bug in the old version),
- is complete (the old one was cut off mid-function and couldn't run),
- returns relative size (fraction of image area + pixel box) rather than a fabricated real-world size,
- computes a per-detection hazard score and an aggregated image-level damage score,
- flags manhole detections as a hard safety flag, using a lower confidence threshold since a missed manhole is worse than an extra false positive.


In [ ]:
from ultralytics import YOLO as _YOLO

CLASS_NAMES = {0: 'pothole', 1: 'crack', 2: 'manhole'}

# Hazard weights — independent of size, tune against your own judgement/field data.
HAZARD_WEIGHT = {'pothole': 2.0, 'crack': 1.0, 'manhole': 3.0}

# Per-class confidence thresholds. Manhole is rare in training data and
# safety-critical if missed, so it gets a lower bar than pothole/crack.
CONF_THRESHOLD = {'pothole': 0.35, 'crack': 0.35, 'manhole': 0.25}


def severity_tier(area_ratio):
    """Bucket by % of the frame the detection covers. These thresholds are a
    starting point — recalibrate once you have real user-uploaded photos,
    since framing distance changes what '% of frame' means physically."""
    if area_ratio < 0.02:
        return 'minor'
    elif area_ratio < 0.08:
        return 'moderate'
    else:
        return 'severe'


def analyze_road_damage_photo(image_path, model):
    """Run detection on a citizen-uploaded photo and return a structured result.

    Returns a dict with:
      - detections: list of {class, confidence, bbox_px, size, severity, hazard_score}
      - manhole_detected: bool safety flag
      - image_damage_score: aggregated severity across all detections (0 if clean)
      - annotated_image_path: path to the image with boxes drawn, for a UI to display
    """
    results = model(image_path, conf=0.20, verbose=False)[0]  # loose global floor;
                                                                # real filtering happens per-class below
    img_h, img_w = results.orig_shape
    img_area = img_w * img_h

    detections = []
    manhole_detected = False

    for box in results.boxes:
        cls_id = int(box.cls[0])
        cls_name = CLASS_NAMES[cls_id]
        conf = float(box.conf[0])

        if conf < CONF_THRESHOLD[cls_name]:
            continue

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        box_w_px, box_h_px = x2 - x1, y2 - y1
        area_ratio = (box_w_px * box_h_px) / img_area

        tier = severity_tier(area_ratio)
        hazard_score = round(area_ratio * HAZARD_WEIGHT[cls_name], 4)

        if cls_name == 'manhole':
            manhole_detected = True

        detections.append({
            'class': cls_name,
            'confidence': round(conf, 3),
            'bbox_px': [round(x1), round(y1), round(x2), round(y2)],
            'size': {
                'width_px': round(box_w_px),
                'height_px': round(box_h_px),
                'frame_area_pct': round(area_ratio * 100, 2)
            },
            'severity': tier,
            'hazard_score': hazard_score
        })

    image_damage_score = round(sum(d['hazard_score'] for d in detections), 4)

    annotated_path = None
    if detections:
        annotated_path = image_path.rsplit('.', 1)[0] + '_annotated.jpg'
        results.save(filename=annotated_path)

    return {
        'image': image_path,
        'detections': detections,
        'manhole_detected': manhole_detected,
        'image_damage_score': image_damage_score,
        'annotated_image_path': annotated_path
    }

## 10. Try it on a sample image

In [ ]:
inference_model = _YOLO(STABLE_WEIGHTS_PATH)

sample_path = f'{OUT}/images/test/' + os.listdir(f'{OUT}/images/test')[0]
result = analyze_road_damage_photo(sample_path, inference_model)
print(json.dumps(result, indent=2))

if result['annotated_image_path']:
    display(IPImage(result['annotated_image_path']))

## A note on real-world size (cm), not just % of frame

Nothing in a single uncalibrated photo tells a model the true physical size of a crack or pothole — a close-up phone shot and a wide dashcam shot of the same 20cm pothole produce completely different pixel sizes. To get real centimeters you need one of:

1. **Known camera height + angle** (you already log ~1.2–1.5m mount height and 10–15° angle in your dataset — if your upload flow captures the same for every user photo, you can back out ground-plane scale with basic projective geometry), or
2. **A reference object of known size in frame** (e.g. ask the user to place a coin or their shoe near the damage), or
3. **Phone AR/depth APIs** (ARKit/ARCore depth) if you're building a mobile app rather than accepting arbitrary uploads.

Any of these is a separate calibration step layered on top of this detector — it's not something the detection model itself can solve, and if it claims to, treat that number as fiction. For now `frame_area_pct` is the honest, defensible metric to build your severity score on.
